# 本模块用来学习Transformer架构的代码实现（Pytorch版本）

![Transformer流程图](../../information/Transformer1.png)

> 以上是Transformer的流程图

In [62]:
# 导入库
import torch  # Pytorch 核心库
import torch.nn as nn  # 模型模块，快速导入各种神经网络模型和损失函数
from torch import optim  # 优化算法
import torch.utils.data as data
import math
import copy

In [63]:
print(torch.cuda.is_available()) # 检查是否有可用的GPU

True


In [64]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # 设置设备为GPU（如果可用）或CPU

In [65]:
# 多头注意力
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model必须能够被num_heads整除"  # 断言，如果维度无法整除头的数量，就会报错，终止程序运行

        self.d_model = d_model  # 模型维度
        self.num_heads = num_heads  # 注意力头的数量
        self.d_k = d_model // num_heads  # 每个头的维度

        # 线性变换层
        self.W_q = nn.Linear(d_model, d_model)  # Q 变换
        self.W_k = nn.Linear(d_model, d_model)  # K 变换
        self.W_v = nn.Linear(d_model, d_model)  # V 变换
        self.W_o = nn.Linear(d_model, d_model)  # output 变换

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        计算缩放点积注意力
        输入形状：
            Q,K,V: (batch_size, num_heads, seq_length, d_k)

        输出形状:(batch_size, num_heads, seq_length, d_k)

        :param Q:
        :param K:
        :param V:
        :param mask:
        :return:
        """

        # 计算注意力分数（Q*K）实际上就是相似度
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 应用掩码
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        # 计算注意力权重(softmax归一化)
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # 对V向量加权求和
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        """
        将输入张量分割成多个头
        输入形状：(batch_size, seq_length, d_model)
        输出形状: (batch_size, num_heads, seq_length, d_k)

        :param x:
        :return:
        """
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        """
        将多个头的输出合并回原始形状
        输入形状: (batch_size, num_heads, seq_length, d_k)
        输出形状: (batch_size, seq_length, d_model)

        :param x:
        :return:
        """
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        """
        前向传播
        输入：QKV: (batch_size, seq_length, d_model)
        输出： (batch_size, seq_length, d_model)


        :param Q:
        :param K:
        :param V:
        :param mask:
        :return:
        """

        # 线性变换并分割多头
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        # 计算注意力
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)

        # 合并多头并输出变换
        output = self.W_o(self.combine_heads(attn_output))
        return output



In [66]:
# 位置前馈网络
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, d_ff)  # 第一层全连接层
        self.fc2 = nn.Linear(d_ff, d_model)  # 第二层全连接层
        self.relu = nn.ReLU()  # 激活函数

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


In [67]:
# 位置编码
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)  # 初始化位置编码矩阵
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)  # 偶数位置使用正弦函数
        pe[:, 1::2] = torch.cos(position * div_term)  # 奇数位置使用余弦函数
        self.register_buffer('pe', pe.unsqueeze(0))  # 注册为缓冲区

    def forward(self, x):
        # 将位置编码添加到输入中
        return x + self.pe[:, :x.size(1)]



In [68]:
# 构建编码器块
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)  # 自注意力机制
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)  # 前馈网络
        self.norm1 = nn.LayerNorm(d_model)  # 层归一化
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)  # Dropout层

    def forward(self, x, mask):
        # 自注意力机制
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))  # 残差连接和归一化层

        # 前馈网络
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))  # 残差连接和归一化层
        return x

In [69]:
# 构建解码器模块
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)  # 自注意力机制
        self.cross_attn = MultiHeadAttention(d_model, num_heads)  # 交叉注意力机制
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)  # 前馈网络
        self.norm1 = nn.LayerNorm(d_model)  # 层归一化
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)  # Dropout

    def forward(self, x, enc_output, src_mask, tgt_mask):
        # 自注意力机制
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))  # 残差连接和层归一化

        # 交叉注意力机制
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))  # 残差连接和层归一化

        # 前馈网络
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))  # 残差连接和层归一化

        return x

In [70]:
# 构建完整的Transformer模型
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        """
        self,
        src_vocab_size: 源语言词汇表大小（比如英文单词的数量）
        tgt_vocab_size: 目标语言词汇表大小（例如要翻译成中文，中文单词的数量）
        d_model: 模型的维度（每个词向量的长度）
        num_heads: 多头注意力的头的数量
        num_layers: 编码器或者解码器堆叠的层数
        d_ff: 前馈网络隐藏层的维度
        max_seq_length: 最大序列的长度
        dropout: Dropout的概率

        """
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(
            src_vocab_size, d_model)  # 编码器词嵌入
        self.decoder_embedding = nn.Embedding(
            tgt_vocab_size, d_model)  # 解码器词嵌入
        self.positional_encoding = PositionalEncoding(
            d_model, max_seq_length=max_seq_length)  # 位置编码

        # 编码器和解码器层
        self.encoder_layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fv = nn.Linear(d_model, tgt_vocab_size)  # 最终的全连接层
        self.dropout = nn.Dropout(dropout)  # Dropout

    def generate_mask(self, src, tgt):
        # 源掩码：屏蔽填充符
        # 形状：（batch_size, 1, 1, seq_length）
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)

        # 目标掩码：屏蔽填充符和未来信息
        # 形状：（batch_size, 1, seq_length, 1）
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        # 生成上三角矩阵
        nopeak_mask = torch.triu(torch.ones(
            1, seq_length, seq_length, device=tgt.device, dtype=torch.bool), diagonal=1)
        tgt_mask = tgt_mask & nopeak_mask  # 合并填充掩码和未来信息掩码
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        # 生成掩码
        src_mask, tgt_mask = self.generate_mask(src, tgt)

        # 编码器部分
        src_embedded = self.dropout(
            self.positional_encoding(self.encoder_embedding(src)))
        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        # 解码器部分
        tgt_embedded = self.dropout(
            self.positional_encoding(self.decoder_embedding(tgt)))
        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        # 最终输出
        output = self.fv(dec_output)
        return output

In [71]:
# 模型的训练

# 设置超参数
src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1

# 初始化模型
transformer = Transformer(src_vocab_size=src_vocab_size, tgt_vocab_size=tgt_vocab_size, d_model=d_model,
                          num_heads=num_heads, num_layers=num_layers, d_ff=d_ff, max_seq_length=max_seq_length, dropout=dropout)

# 生成随机数据
src_data = torch.randint(1, src_vocab_size, (64, max_seq_length))  # 源序列
tgt_data = torch.randint(1, src_vocab_size, (64, max_seq_length))  # 目标序列

# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss(ignore_index=0)  # 忽略填充部分的损失
optimizer = optim.Adam(transformer.parameters(),
                       lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer = transformer.to(device) # 将模型移动到GPU（如果可用）
src_data = src_data.to(device) # 将输入数据移动到GPU（如果可用）
tgt_data = tgt_data.to(device) # 将目标数据移动到GPU（如果可用）

# 训练循环
transformer.train()
for epoch in range(100):
    optimizer.zero_grad()  # 清空梯度，防止累积

    # 输入目标序列时去掉最后一个词（用于预测下一个词）
    output = transformer(src_data, tgt_data[:, :-1])

    # 计算损失时，目标序列从第二个词开始
    # output形状（batch_size, seq_llength-1, tgt_vocab_size)
    # 目标形状(batch_size, seq_length -1)
    loss = criterion(
        output.contiguous().view(-1, tgt_vocab_size),
        tgt_data[:, 1:].contiguous().view(-1)
    )

    loss.backward()  # 反向传播
    optimizer.step()  # 更新参数
    print(f"Epoch:{epoch+1}, Loss:{loss.item()}")

Epoch:1, Loss:8.689630508422852
Epoch:2, Loss:8.550801277160645
Epoch:3, Loss:8.479141235351562
Epoch:4, Loss:8.425531387329102
Epoch:5, Loss:8.366921424865723
Epoch:6, Loss:8.304082870483398
Epoch:7, Loss:8.224591255187988
Epoch:8, Loss:8.14768123626709
Epoch:9, Loss:8.063972473144531
Epoch:10, Loss:7.987305164337158
Epoch:11, Loss:7.903768539428711
Epoch:12, Loss:7.822312355041504
Epoch:13, Loss:7.740636348724365
Epoch:14, Loss:7.6608195304870605
Epoch:15, Loss:7.579316139221191
Epoch:16, Loss:7.49668550491333
Epoch:17, Loss:7.411203861236572
Epoch:18, Loss:7.332574367523193
Epoch:19, Loss:7.243595123291016
Epoch:20, Loss:7.163354873657227
Epoch:21, Loss:7.0836944580078125
Epoch:22, Loss:7.008635997772217
Epoch:23, Loss:6.932371616363525
Epoch:24, Loss:6.860666751861572
Epoch:25, Loss:6.774689674377441
Epoch:26, Loss:6.698808193206787
Epoch:27, Loss:6.626472473144531
Epoch:28, Loss:6.55203914642334
Epoch:29, Loss:6.489513874053955
Epoch:30, Loss:6.420374393463135
Epoch:31, Loss:6.339

In [ ]:
# 模型评估
transformer.eval()
# 生成验证数据
val_src_data = torch.randint(1, src_vocab_size, (64, max_seq_length)).to(device)
val_tgt_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length)).to(device)
# 假设输入为一批英文和对应的中文翻译（已转换为索引）
# 示例数据：
# src_data: [[3, 14, 25, ..., 0, 0], ...] # 英文句子（0为填充符）
# tgt_data: [[5, 20, 36, ..., 0, 0], ...] # 中文翻译（0为填充符）
# 注意：实际应用中需对文本进行分词、编码、填充等预处理
with torch.no_grad():
    val_output = transformer(val_src_data, val_tgt_data[:, :-1])
    val_loss = criterion(val_output.contiguous().view(-1, tgt_vocab_size), val_tgt_data[:, 1:].contiguous().view(-1))
print(f"Validation Loss: {val_loss.item()}")    


Validation Loss: 8.79386043548584
